# 01 · Inflación vs. dólar en Argentina
Primer análisis del laboratorio de datos. Usamos la **API de Series de Tiempo** (apis.datos.gob.ar) para:
1. Buscar series por palabra clave.
2. Descargar el IPC y el tipo de cambio.
3. Guardarlos y consultarlos con SQL (DuckDB).
4. Graficar y sacar conclusiones.

> En Colab: *Entorno de ejecución → Ejecutar todas*.

## 0. Preparación
Instalamos lo que falta y cargamos las funciones para hablar con la API.

In [ ]:
!pip install -q duckdb plotly

In [ ]:
"""
datosar.py — Funciones para buscar y descargar series de la
API de Series de Tiempo de Argentina (apis.datos.gob.ar/series).

Uso rápido:
    from datosar import buscar_series, traer_series
    buscar_series("tipo de cambio")
    df = traer_series(["168.1_T_CAMBIOR_D_0_0_26"], frecuencia="month")
"""

import io
import requests
import pandas as pd

BASE = "https://apis.datos.gob.ar/series/api"


def buscar_series(texto: str, limite: int = 20) -> pd.DataFrame:
    """Busca series por palabras clave y devuelve una tabla con id, descripción y datos útiles."""
    r = requests.get(f"{BASE}/search/", params={"q": texto, "limit": limite}, timeout=30)
    r.raise_for_status()
    filas = []
    for item in r.json().get("data", []):
        campo = item.get("field", {})
        dataset = item.get("dataset", {})
        filas.append({
            "id": campo.get("id"),
            "descripcion": campo.get("description"),
            "unidades": campo.get("units"),
            "frecuencia": campo.get("frequency"),
            "desde": campo.get("time_index_start"),
            "hasta": campo.get("time_index_end"),
            "dataset": dataset.get("title"),
            "fuente": dataset.get("source"),
        })
    return pd.DataFrame(filas)


def traer_series(ids, frecuencia: str | None = None, agregacion: str = "avg",
                 modo: str | None = None, desde: str | None = None) -> pd.DataFrame:
    """
    Descarga una o más series y las devuelve en un DataFrame con la fecha como índice.

    frecuencia: None (original), "month", "quarter" o "year"
    agregacion: cómo agrupar al cambiar la frecuencia ("avg", "sum", "end_of_period", ...)
    modo:       None (valores), "percent_change" (var. % vs. período anterior)
                o "percent_change_a_year_ago" (var. % interanual)
    desde:      fecha inicial, ej. "2017-01-01"
    """
    if isinstance(ids, str):
        ids = [ids]
    params = {"ids": ",".join(ids), "format": "csv", "limit": 5000}
    if frecuencia:
        params["collapse"] = frecuencia
        params["collapse_aggregation"] = agregacion
    if modo:
        params["representation_mode"] = modo
    if desde:
        params["start_date"] = desde
    r = requests.get(f"{BASE}/series/", params=params, timeout=60)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text), parse_dates=["indice_tiempo"])
    return df.set_index("indice_tiempo")


# Series de ejemplo (si alguna dejara de actualizarse, buscá su reemplazo con buscar_series)
IPC_NACIONAL = "148.3_INIVELNAL_DICI_M_26"       # IPC nivel general, total nacional
DOLAR_BNA = "168.1_T_CAMBIOR_D_0_0_26"           # Tipo de cambio BNA, vendedor, diario


## 1. Buscar series
Cambiá el texto para explorar otras. La columna `id` es la que se usa para descargar.

In [ ]:
resultados = buscar_series("ipc nivel general nacional")
resultados[["id", "descripcion", "frecuencia", "desde", "hasta", "fuente"]]

In [ ]:
buscar_series("tipo de cambio bna")[["id", "descripcion", "frecuencia", "desde", "hasta"]]

## 2. Descargar los datos
Pasamos el dólar (diario) a promedio mensual para que coincida con el IPC.

Si alguna serie viene vacía o desactualizada, buscá el `id` nuevo en el paso 1 y reemplazalo acá.

In [ ]:
df = traer_series([IPC_NACIONAL, DOLAR_BNA], frecuencia="month", desde="2017-01-01")
df.columns = ["ipc", "dolar"]
df.tail()

## 3. Limpiar y revisar
Siempre mirar: cuántos datos hay, si faltan valores y el rango de fechas.

In [ ]:
print("Filas:", len(df))
print("Desde", df.index.min().date(), "hasta", df.index.max().date())
print("\nValores faltantes por columna:")
print(df.isna().sum())
df = df.dropna()
df.describe()

## 4. Guardar y consultar con SQL (DuckDB)
DuckDB permite hacer consultas SQL directamente sobre un DataFrame o un archivo, sin instalar un servidor.

In [ ]:
import duckdb
df.reset_index().to_csv("ipc_dolar_mensual.csv", index=False)

con = duckdb.connect()
anual = con.sql("""
    SELECT year(indice_tiempo) AS anio,
           round(last(ipc ORDER BY indice_tiempo) / first(ipc ORDER BY indice_tiempo) * 100 - 100, 1) AS inflacion_pct,
           round(last(dolar ORDER BY indice_tiempo) / first(dolar ORDER BY indice_tiempo) * 100 - 100, 1) AS suba_dolar_pct
    FROM 'ipc_dolar_mensual.csv'
    GROUP BY anio
    ORDER BY anio
""").df()
anual

*Nota:* la variación se calcula entre el primer y el último mes de cada año, así que es una aproximación a la inflación anual oficial (que compara diciembre contra diciembre).

## 5. Graficar
Como el IPC y el dólar tienen escalas distintas, los llevamos a una base común: **primer mes = 100**.

In [ ]:
import plotly.express as px

base100 = df / df.iloc[0] * 100
fig = px.line(base100, labels={"value": "Índice (base = 100)", "indice_tiempo": "Fecha", "variable": "Serie"},
              title="Precios vs. dólar oficial (base enero 2017 = 100)")
fig.update_layout(hovermode="x unified")
fig.show()

In [ ]:
fig2 = px.bar(anual, x="anio", y=["inflacion_pct", "suba_dolar_pct"], barmode="group",
              labels={"value": "%", "anio": "Año", "variable": ""},
              title="Inflación vs. suba del dólar oficial, por año")
fig2.show()

## 6. Un indicador propio: el dólar en términos reales
Si el dólar sube menos que los precios, se *abarata* en términos reales. Si sube más, se encarece.

In [ ]:
df["dolar_real"] = df["dolar"] / df["ipc"] * df["ipc"].iloc[-1]   # en pesos de hoy
px.line(df, y="dolar_real", title="Dólar oficial a precios de hoy",
        labels={"dolar_real": "$ de hoy", "indice_tiempo": "Fecha"}).show()

## 7. Conclusiones
Escribí acá, con tus palabras, qué muestran los gráficos:
- ¿En qué años el dólar subió más que los precios, y en cuáles menos?
- ¿Cuándo estuvo más "caro" o más "barato" el dólar en términos reales?
- ¿Qué otra serie te gustaría cruzar? (salarios, gas, combustibles…)

## Próximos pasos
- Probar `traer_series(..., modo="percent_change_a_year_ago")` para ver variaciones interanuales directo de la API.
- Sumar una serie de salarios (`buscar_series("indice de salarios")`).
- Guardar este notebook en GitHub dentro de `notebooks/`.